### Connexion à la DB DuckDB

In [3]:
import duckdb
import os
from pathlib import Path
from typing import List
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
from tqdm import tqdm
import sklearn
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

### Connexion à la DB / Import des Data


In [15]:
# Store database at project root
DB_NAME = Path("/home/c-enjalbert/Documents/EPSI/MSPR/bloc_2/amazing/amazing.duckdb") 
# Go up one level from current directory to get to project root
data_folder = Path("..") / "data"
# For absolute certainty, you could use the absolute path
# data_folder = Path("/home/c-enjalbert/Documents/EPSI/MSPR/bloc_2/amazing/data")
con = duckdb.connect(str(DB_NAME))

In [16]:
# 2. Query to list all tables in the database
# DuckDB specific way to list tables
tables_info = con.sql("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
""").df()

print(f"Found {len(tables_info)} tables in the database:\n")

if len(tables_info) > 0:
    for i, table_name in enumerate(tables_info['table_name']):
        print(f"{i+1}. {table_name}")
else:
    print("No tables found in the database.")

Found 4 tables in the database:

1. all_events
2. loaded_files
3. user_events
4. user_segments


In [17]:
# 5. Alternative way to show all tables
print("List of all tables using DuckDB's connections.tables():")
con.sql("SHOW TABLES").show()

List of all tables using DuckDB's connections.tables():
┌───────────────┐
│     name      │
│    varchar    │
├───────────────┤
│ all_events    │
│ loaded_files  │
│ user_events   │
│ user_segments │
└───────────────┘



In [18]:
# Examine the all_events table
print("First 10 rows of user_events table:")
all_events_data = con.sql("""
    SELECT * FROM user_events LIMIT 10
""")
all_events_data.show()


# Examine the loaded_files table
print("\nContents of loaded_files table:")
loaded_files_data = con.sql("""
    SELECT * FROM loaded_files
""")
loaded_files_data.show()


First 10 rows of user_events table:
┌───────────┬──────────────┬─────────────┬─────────────────┬─────────────────────────┬────────────────────┬────────────────────┬─────────────────────┬─────────────────────┬────────────────────┬───────────────────────┐
│  user_id  │ total_events │ total_views │ total_purchases │ avg_time_between_events │    total_spent     │     avg_basket     │   last_event_time   │   conversion_rate   │   purchase_ratio   │ days_since_last_event │
│  varchar  │    int64     │   double    │     double      │         double          │       double       │       double       │      timestamp      │       double        │       double       │         int64         │
├───────────┼──────────────┼─────────────┼─────────────────┼─────────────────────────┼────────────────────┼────────────────────┼─────────────────────┼─────────────────────┼────────────────────┼───────────────────────┤
│ 603627806 │          182 │       181.0 │             0.0 │       8467.906077348067 │      

In [7]:
# Examine the all_events table
print("First 10 rows of all_events table:")
all_events_data = con.sql("""
    SELECT * FROM all_events LIMIT 10
""")
all_events_data.show()

# Show count of records in all_events
record_count = con.sql("""
    SELECT COUNT(*) as total_events FROM all_events
""")
record_count.show()

# Examine the loaded_files table
print("\nContents of loaded_files table:")
loaded_files_data = con.sql("""
    SELECT * FROM loaded_files
""")
loaded_files_data.show()


First 10 rows of all_events table:
┌─────────────────────┬────────────┬────────────┬─────────────────────┬──────────────────────────────────┬─────────┬────────┬───────────┬──────────────────────────────────────┐
│     event_time      │ event_type │ product_id │     category_id     │          category_code           │  brand  │ price  │  user_id  │             user_session             │
│      timestamp      │  varchar   │  varchar   │       varchar       │             varchar              │ varchar │ double │  varchar  │               varchar                │
├─────────────────────┼────────────┼────────────┼─────────────────────┼──────────────────────────────────┼─────────┼────────┼───────────┼──────────────────────────────────────┤
│ 2020-02-01 00:00:00 │ view       │ 100010426  │ 2232732115290555244 │ apparel.shoes                    │ NULL    │  19.31 │ 581521975 │ 82d5664d-7980-43e1-ab27-a2f05e991760 │
│ 2020-02-01 00:00:00 │ view       │ 100071077  │ 2232732079706079299 │ sport.bi

In [8]:
DB_NAME = "amazing.duckdb"
TABLE_EVENTS = "all_events"
TABLE_USER_EVENTS = "user_events"
SAMPLE_USER_PERCENT = 0.005
BATCH_SIZE = 1000 

### Import de la table DuckDB

In [9]:
# Chargement de users avec au moins 10 événements 
print("Chargement d'un échantillon d'utilisateurs actifs...")

# Afficher les tables disponibles dans la base de données
tables_df = con.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'main'").fetch_df()
print("Tables disponibles dans la base de données :")
print(tables_df)

user_ids_df = con.execute(f"""
    SELECT user_id
    FROM all_events
    WHERE user_id IS NOT NULL
    GROUP BY user_id
    HAVING COUNT(*) >= 10
""").fetch_df()

Chargement d'un échantillon d'utilisateurs actifs...
Tables disponibles dans la base de données :
      table_name
0     all_events
1   loaded_files
2    user_events
3  user_segments


### Normalisation et Standardisation des données

In [10]:


sampled_user_ids = user_ids_df.sample(frac=SAMPLE_USER_PERCENT, random_state=42)['user_id'].tolist()

print(f"Nombre d'utilisateurs actifs échantillonnés : {len(sampled_user_ids)}")

#  Création des features utilisateurs batch par batch 
print("Création des features utilisateurs par batch...")

user_features_list = []

for i in tqdm(range(0, len(sampled_user_ids), BATCH_SIZE), desc="Avancement user features", ncols=100):
    batch_ids = sampled_user_ids[i:i+BATCH_SIZE]
    batch_ids_str = ",".join(f"'{uid}'" for uid in batch_ids)

    batch_query = f"""
    WITH
        base_events AS (
            SELECT
                user_id,
                event_type,
                event_time,
                price,
                LEAD(event_time) OVER (PARTITION BY user_id ORDER BY event_time) AS next_event_time
            FROM {TABLE_EVENTS}
            WHERE user_id IN ({batch_ids_str})
        ),
        features AS (
            SELECT
                user_id,
                COUNT(*) AS total_events,
                SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS total_views,
                SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS total_purchases,
                AVG(EXTRACT(EPOCH FROM (next_event_time - event_time))) AS avg_time_between_events,
                SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END) AS total_spent,
                COALESCE(AVG(CASE WHEN event_type = 'purchase' THEN price ELSE NULL END), 0) AS avg_basket,
                MAX(event_time) AS last_event_time
            FROM base_events
            GROUP BY user_id
    )
    SELECT
        *,
        CASE WHEN total_views > 0 THEN total_purchases * 1.0 / total_views ELSE 0 END AS conversion_rate,
        CASE WHEN (total_views + total_purchases) > 0 THEN total_purchases * 1.0 / (total_views + total_purchases) ELSE 0 END AS purchase_ratio,
        DATE_PART('day', CAST('2020-03-31 22:00:00' AS TIMESTAMP) - last_event_time) AS days_since_last_event
    FROM features
    """

    batch_features = con.execute(batch_query).fetch_df()

    # Récupérer les user_id de cette batch
    valid_user_ids = con.execute(f"""
        SELECT user_id
        FROM {TABLE_EVENTS}
        WHERE user_id IN ({batch_ids_str})
        GROUP BY user_id
        HAVING COUNT(*) >= 10
    """).fetch_df()
    valid_user_ids = set(valid_user_ids["user_id"].astype(str))

    # Filtrage strict des user_id valides
    batch_features = batch_features[batch_features["user_id"].astype(str).isin(valid_user_ids)]

    user_features_list.append(batch_features)

# Fusionner tous les batchs
user_features = pd.concat(user_features_list, ignore_index=True)

# Vérification des NaN
print("Vérification des NaN")
nan_summary = user_features.isna().sum()
print("Résumé des NaN par colonne :")
print(nan_summary[nan_summary > 0])

users_with_nan = user_features[user_features.isna().any(axis=1)]
print(f"Nombre d'utilisateurs avec des NaN : {len(users_with_nan)}")
print("Exemples d'utilisateurs avec NaN :")
print(users_with_nan.head(10))

# Standardisation
print("Standardisation des features...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(user_features.drop(columns=["last_event_time"]))



Nombre d'utilisateurs actifs échantillonnés : 5989
Création des features utilisateurs par batch...


Avancement user features: 100%|███████████████████████████████████████| 6/6 [00:03<00:00,  1.61it/s]

Vérification des NaN
Résumé des NaN par colonne :
Series([], dtype: int64)
Nombre d'utilisateurs avec des NaN : 0
Exemples d'utilisateurs avec NaN :
Empty DataFrame
Columns: [user_id, total_events, total_views, total_purchases, avg_time_between_events, total_spent, avg_basket, last_event_time, conversion_rate, purchase_ratio, days_since_last_event]
Index: []
Standardisation des features...


In [11]:
user_features

,user_id,total_events,total_views,total_purchases,avg_time_between_events,total_spent,avg_basket,last_event_time,conversion_rate,purchase_ratio,days_since_last_event
0,603627806,182,181.0,0.0,8467.906077,0.00,0.000000,2020-02-19 03:27:45,0.000000,0.000000,41
1,605007025,88,85.0,0.0,5307.344828,0.00,0.000000,2020-02-09 17:34:13,0.000000,0.000000,51
2,604928591,123,88.0,0.0,14570.016393,0.00,0.000000,2020-02-23 15:40:55,0.000000,0.000000,37
3,517754395,45,42.0,0.0,19462.477273,0.00,0.000000,2020-02-25 18:38:04,0.000000,0.000000,35
4,593005978,13,7.0,3.0,38297.083333,1719.32,573.106667,2020-02-20 12:12:08,0.428571,0.300000,40
...,...,...,...,...,...,...,...,...,...,...,...
5984,613502458,16,16.0,0.0,2909.266667,0.00,0.000000,2020-02-18 04:29:46,0.000000,0.000000,42
5985,563583245,48,37.0,0.0,16479.170213,0.00,0.000000,2020-02-18 16:31:55,0.000000,0.000000,42
5986,567845212,40,37.0,1.0,10962.948718,72.51,72.510000,2020-02-27 06:01:03,0.027027,0.026316,33
5987,527618969,19,19.0,0.0,107446.444444,0.00,0.000000,2020-02-28 15:02:22,0.000000,0.000000,32


In [12]:
#  Sauvegarde des résultats dans DuckDB
print(f"Sauvegarde dans {TABLE_USER_EVENTS}...")
con.execute(f"DROP TABLE IF EXISTS {TABLE_USER_EVENTS}")
con.register("temp_user_features", user_features)
con.execute(f"CREATE TABLE {TABLE_USER_EVENTS} AS SELECT * FROM temp_user_features")

Sauvegarde dans user_events...


In [13]:
con.close()